## **Flipkart Data Pipeline (Medallion Architecture)
## 1. SAMPLE DATASET (RAW - BRONZE INPUT)**

In [0]:
%python
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()


data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

df = spark.createDataFrame(data, columns)

df.display()



order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df.write.mode("append").format("delta").save('/Volumes/workspace/default/med_architecture/bronze_data')

In [0]:
df = spark.read.format("delta").load('/Volumes/workspace/default/med_architecture/bronze_data')
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


## 2. SILVER LAYER (CLEANED DATA)

In [0]:
## Handle Null values
df = df.fillna({
    "amount" :"0",
    "city" :"unknown"
})
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,unknown,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
## Fix Data Types
from pyspark.sql.functions import *;

df = df.withColumn("amount", col("amount").cast("int"))
df = df.withColumn("date", to_date(col("date")))
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,unknown,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
## Remove Negative Value
df = df.filter(col("amount") > 0)
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1


In [0]:
## Handle Duplicates + Updates
from pyspark.sql.functions import *
from pyspark.sql.window import Window


window  = Window.partitionBy("order_id").orderBy(col("date").desc())
df = df.withColumn("rn", row_number().over(window)) \
       .filter(col("rn") == 1).drop("rn")
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
## Standardization (Optional)
df =df.withColumn("city",initcap(col("city")))
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df.write.mode("overwrite").format("delta").save('/Volumes/workspace/default/med_architecture/silver_data')

In [0]:
## Read the loaded data
df_silver_data = spark.read.format('delta').load('/Volumes/workspace/default/med_architecture/silver_data')

df_silver_data.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df_gold_data = spark.read.format("delta").load('/Volumes/workspace/default/med_architecture/gold_data')
df_gold_data.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df_silver_data.write.mode("overwrite").format("delta").save('/Volumes/workspace/default/med_architecture/gold_data')

## GOLD LAYER (BUSINESS DATA)

In [0]:
##  Requirement 1: Sales Analysis
## Total Sales by Product

df_gold_data.groupBy("product")\
    .sum("amount")\
    .withColumnRenamed("sum(amount)","total_product_sales").display()

product,total_product_sales
Tablet,22000
Mobile,33000
Watch,8000
Laptop,105000
Headphones,3000


In [0]:
## Total Sales by Category
df_gold_data.groupBy("category")\
    .sum("amount")\
        .withColumnRenamed("sum(amount)","total_category_sales").display()
                           

category,total_category_sales
Electronics,160000
Accessories,11000


In [0]:
## Total Sales by City
df_gold_data.groupBy("city")\
    .sum("amount")\
        .withColumnRenamed("sum(amount)","total_city_sales").display()

city,total_city_sales
Delhi,55000
Chennai,33000
Unknown,3000
Hyderabad,72000
Mumbai,8000


In [0]:
## Requirement 2: Customer Insights
## Orders per Customer

df_gold_data.groupBy("customer_id").count().display()



customer_id,count
C005,1
C004,1
C003,1
C001,2
C002,1
C007,1


In [0]:
## Total Spending per Customer
df_gold_data.groupBy("customer_id")\
    .sum("amount").display()

customer_id,sum(amount)
C005,3000
C004,8000
C003,55000
C001,72000
C002,18000
C007,15000


In [0]:
## Requirement 3: Top Analysis
## Top Customer
df_gold_data.groupBy("customer_id") \
  .sum("amount") \
  .orderBy(col("sum(amount)").desc()).display()

customer_id,sum(amount)
C001,72000
C003,55000
C002,18000
C007,15000
C004,8000
C005,3000


In [0]:
## Top Product
df_gold_data.groupBy("product")\
    .sum("amount")\
        .orderBy(sum("amount").desc()).display()

product,sum(amount)
Laptop,105000
Mobile,33000
Tablet,22000
Watch,8000
Headphones,3000
